# Module 5 — Survival Analysis

**Biological question:** Do PAM50 molecular subtypes predict how long a breast cancer patient survives? If subtypes differ in prognosis, this confirms that our molecular classification captures clinically relevant biology — and motivates building a molecular prognostic signature in Module 8.

**Methods:**
- **Kaplan-Meier curves** — non-parametric estimator of the survival function over time. One curve per PAM50 subtype; at-risk counts shown below. A curve that drops faster indicates worse overall survival.
- **Log-rank test** — pairwise hypothesis test: are two KM curves statistically distinguishable? p < 0.05 means the two subtypes have significantly different survival.
- **Cox proportional hazards model** — multivariate model that estimates hazard ratios (HR) for each subtype relative to a reference. HR > 1 means higher instantaneous hazard of death; HR < 1 means lower.
- **ESR1 expression split** — ESR1 (oestrogen receptor 1) is the canonical marker of luminal breast cancer and is known to be prognostic. We validate this by splitting patients into ESR1-high vs ESR1-low groups.

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.constants import DataFile, InterimFile
from src.paths import interim_file, raw_file
from src.models.survival import merge_survival, fit_km, logrank_pairwise, fit_cox
from src.visualization.survival_plots import plot_km_by_subtype, plot_cox_forest

logging.basicConfig(level=logging.INFO, format="%(message)s")

## Load and Merge Survival Data

In [ ]:
labels = pd.read_csv(interim_file(InterimFile.LABELS), index_col=0).squeeze()
surv_raw = pd.read_csv(raw_file(DataFile.SURVIVAL), sep="\t", index_col=0)

df = merge_survival(labels=labels, raw_survival_df=surv_raw)
print(f"Merged: {len(df)} patients with OS data")
print(df["subtype"].value_counts())

out = interim_file(InterimFile.SURVIVAL_MERGED)
df.to_csv(out)
print(f"Saved {out.name}")

## Kaplan-Meier Curves by PAM50 Subtype

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_km_by_subtype(df=df, ax=ax)
plt.tight_layout()
plt.show()

## Pairwise Log-Rank P-Value Heatmap

Low p-values (dark cells) indicate subtype pairs whose survival curves are statistically distinguishable. The Basal subtype is expected to differ significantly from the luminal subtypes.

In [ ]:
p_mat = logrank_pairwise(df)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    np.log10(p_mat + 1e-300) * -1,
    annot=p_mat.round(3),
    fmt=".3f",
    cmap="YlOrRd",
    ax=ax,
    cbar_kws={"label": "-log10(p-value)"},
)
ax.set_title("Pairwise log-rank p-values", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

## Cox Proportional Hazards Model — PAM50 Subtype

We use PAM50 subtype as a single categorical covariate. The forest plot shows hazard ratios for each subtype dummy variable relative to the dropped reference category (alphabetically first: Basal). HR > 1 means that subtype has a higher hazard than Basal.

In [ ]:
cph = fit_cox(df=df, covariates=["subtype"])
print(f"Concordance index: {cph.concordance_index_:.3f}")
cph.print_summary()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_cox_forest(cox_fitter=cph, ax=ax)
plt.tight_layout()
plt.show()

## ESR1 Expression Split — Survival Curve

ESR1 (oestrogen receptor 1) is the canonical marker of luminal breast cancer. High ESR1 expression predicts better overall survival — this is the molecular basis for endocrine therapy. We split patients at the median ESR1 log-expression and compare survival.

The Ensembl base ID for ESR1 is `ENSG00000091831`. We match against the versioned column names in X.

In [ ]:
X = pd.read_csv(interim_file(InterimFile.X), index_col=0)

ESR1_BASE = "ENSG00000091831"
esr1_cols = [c for c in X.columns if c.startswith(ESR1_BASE)]

if esr1_cols:
    esr1_expr = X[esr1_cols[0]]
    shared = df.index.intersection(esr1_expr.index)
    df_esr1 = df.loc[shared].copy()
    df_esr1["ESR1_group"] = pd.cut(
        esr1_expr.loc[shared],
        bins=[esr1_expr.loc[shared].min() - 0.001, esr1_expr.loc[shared].median(), esr1_expr.loc[shared].max()],
        labels=["ESR1 low", "ESR1 high"],
    )

    from lifelines import KaplanMeierFitter
    from lifelines.plotting import add_at_risk_counts

    fig, ax = plt.subplots(figsize=(9, 6))
    fitters = []
    for group in ["ESR1 high", "ESR1 low"]:
        sub = df_esr1.loc[df_esr1["ESR1_group"] == group]
        kmf = KaplanMeierFitter(label=group)
        kmf.fit(durations=sub["OS_time"], event_observed=sub["OS_event"])
        kmf.plot_survival_function(ax=ax, ci_show=True)
        fitters.append(kmf)
    add_at_risk_counts(*fitters, ax=ax)
    ax.set_title("ESR1 expression: high vs low survival", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Survival probability")
    ax.legend(fontsize=9, frameon=False)
    plt.tight_layout()
    plt.show()
else:
    print(f"ESR1 ({ESR1_BASE}) not found in filtered gene set — was likely removed during variance filtering.")